In [3]:
import time

from langchain.chat_models import init_chat_model

from langchain_demo.config import load_project_environment, require_environment_variable

load_project_environment(override=True)
api_key = require_environment_variable("DEEPSEEK_API_KEY")
base_url = require_environment_variable("DEEPSEEK_API_BASE")

model = init_chat_model(
    model="deepseek:deepseek-flash",  # 最好明确指定供应商
    api_key=api_key,
    base_url=base_url,
)

## 流式输出

* 响应更快，用户不必等待完整输出
* 交互体验会更流畅，尤其是在长文本情况下
* 可以实时展示模型的思考过程


In [ ]:
for chunk in model.stream("解释下什么是人工智能"):
    print(
        chunk.text, end="", flush=True
    )  #  flush=True,没收到一个chunk,立刻刷新输出，而不是将数据存在内存缓冲区，等缓冲区满或程序结束后一次性输出

### 批量调用
* 一次性接收所有的响应

In [ ]:
message = ["who are you?", "where are you from?"]

responses = model.batch(message)
for response in responses:
    print(response.text, flush=True)

* 按完成的顺序接收响应

In [ ]:
message = ["请介绍一下你自己", "1+1=?"]
responses = model.batch_as_completed(message)
for response in responses:
    # responses是一个元祖，元祖的第一个元素是问题的索引下标，结果根据完成时间顺序重新排序
    print(response[1].content, flush=True)

* 批量调用和按完成顺序调用耗时对比

In [4]:
messages = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天凉爽",
    "翻译成英文：冬天寒冷",
]

start_time = time.time()
responses = model.batch(messages)
for response in responses:
    print(response.text, flush=True)
end_time = time.time() - start_time
print(end_time, "seconds")

Spring has arrived.
Summer is very hot.
Autumn is cool.
Winter is cold.
1.759127140045166 seconds


In [ ]:
messages = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天凉爽",
    "翻译成英文：冬天寒冷",
]

start_time = time.time()
for message in messages:
    response = model.invoke(message)
    print(response.content)
end_time = time.time() - start_time
print(end_time, "seconds")